In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_13_try_0.pickle

In [ ]:
%%RecordEvent
%%time
### cell 14 ###

# Preprocess 2018 and 2019 datasets without modifying the originals
orig_q = 'On which platforms have you begun or completed data science courses'
alt_q = 'On which online platforms have you begun or completed data science courses'

col_map_2018 = {
    alt_q: orig_q,
    'Kaggle Learn': 'Kaggle Learn Courses',
    'Fast.AI': 'Fast.ai',
    'Online University Courses': 'University Courses (resulting in a university degree)'
}
val_map_2018 = {
    'Kaggle Learn': 'Kaggle Learn Courses',
    'Fast.AI': 'Fast.ai',
    'Online University Courses': 'University Courses (resulting in a university degree)'
}
col_map_2019 = {'Kaggle Courses (i.e. Kaggle Learn)': 'Kaggle Learn Courses'}
val_map_2019 = col_map_2019

# Rename only once per DataFrame
# 2018
_df = responses_df_2018_cell10.copy()
cols = _df.columns
for old, new in col_map_2018.items():
    cols = cols.str.replace(old, new, regex=False)
_df.columns = cols
# 2019
_df2 = responses_df_2019_cell10.copy()
cols2 = _df2.columns
for old, new in col_map_2019.items():
    cols2 = cols2.str.replace(old, new, regex=False)
_df2.columns = cols2

# Helper to grab, remap and clean only the question-specific columns
def grab_subset(df, question, vmap=None):
    # select columns containing the question
    mask = df.columns.str.contains(question)
    sub = df.loc[:, mask]
    if vmap:
        # restrict replace to just these columns
        sub = sub.replace(vmap)
    # extract labels after the last '-' and strip leading spaces
    labels = sub.columns.str.split('-', n=1).str[-1].str.lstrip()
    sub.columns = labels
    # drop respondents who didn't select any of these
    return sub.dropna(how='all', subset=labels)

# Combine and count per year
def combine_data(question, include_2017=False):
    years = ['2018','2019','2020','2021','2022']
    dfs   = [_df, _df2, responses_df_2020, responses_df_2021, responses_df_2022_cell10]
    vmaps = [val_map_2018, val_map_2019, None, None, None]
    if include_2017:
        years.insert(0, '2017')
        dfs.insert(0, responses_df_2017)
        vmaps.insert(0, None)
    parts = [
        grab_subset(df, question, vmap).assign(year=yr)
        for df, yr, vmap in zip(dfs, years, vmaps)
    ]
    combined = pd.concat(parts, ignore_index=True)
    counts = combined.groupby('year', as_index=False).count()
    return combined, counts

# Convert counts to percentages
def to_percentages(combined, counts):
    totals = combined['year'].value_counts()
    pct = counts.set_index('year').div(totals, axis=0) * 100
    return pct.reset_index()

# Run the pipeline
question_of_interest_cell26 = orig_q + '?'
learning_platform_df_combined, learning_platform_df_combined_counts = (
    combine_data(question_of_interest_cell26)
)
learning_platform_df_combined_percentages = to_percentages(
    learning_platform_df_combined,
    learning_platform_df_combined_counts
)
# Clean up column names
learning_platform_df_combined_percentages.columns = (
    learning_platform_df_combined_percentages.columns
    .str.replace('(resulting in a university degree)', '', regex=False)
)
# Select and melt only the desired platforms
keep_cols = [
    'year', 'Coursera', 'University Courses ', 'Kaggle Learn Courses',
    'Udemy', 'Udacity', 'DataCamp', 'edX', 'Fast.ai', 'None', 'Other'
]
subset = learning_platform_df_combined_percentages.loc[:, keep_cols]
value_vars = [c for c in subset.columns if c not in ('year', 'None', 'Other')]
df_cell26 = (
    subset
    .melt(id_vars='year', value_vars=value_vars)
    .sort_values(['year', 'value'])
    .rename(columns={'variable': ''})
)
df_cell26.info()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_14_try_1.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_14_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[14], f)


In [ ]:
opt_output = Out.get(4)